<!-- CODEx bilingual cell explanation: start -->
### Cell 01 — 环境、路径与 SRC 参数 / Environment, paths, and SRC settings

**中文说明**：导入 SRC、交叉验证、标准化和绘图所需库，建立 Step 2 输出目录，设置目标变量、随机种子和快速验证开关。BOOTSTRAP_N 控制 bootstrap 次数，默认用于论文级结果，快速模式用于本地 smoke test。

**输入与依赖**：依赖本地 Python/Jupyter 环境、项目根目录和后续单元需要共享的配置参数。

**主要输出**：建立路径、随机种子、绘图样式、配置字典或输出目录等基础对象。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Imports the libraries required for SRC, cross-validation, standardisation, and plotting; creates Step 2 output directories; and defines the target, random seed, and fast-validation switch. BOOTSTRAP_N controls bootstrap repetitions, with the default intended for manuscript-grade results and fast mode for local smoke tests.

**Inputs and dependencies**: Depends on the local Python/Jupyter environment, the project root, and shared configuration values used by later cells.

**Main outputs**: Creates paths, random seeds, plotting defaults, configuration dictionaries, or output directories.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:

from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import StandardScaler

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "grid.linestyle": "--",
})

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "input" / "Beijing.epw").exists() or (candidate / "data" / "step1_simulation_dataset.csv").exists():
            return candidate
        if (candidate / "AGENTS.md").exists() and (candidate / "outputs_step1").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
CODE_DIR = PROJECT_ROOT / "experiment_code"
DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "outputs_step2"
FIG_DIR = OUT_DIR / "figures"
PAPER_ASSETS_DIR = PROJECT_ROOT / "paper_assets"
PAPER_ASSETS_FIG_DIR = PAPER_ASSETS_DIR / "figures"
for search_path in [PROJECT_ROOT, CODE_DIR]:
    if search_path.exists() and str(search_path) not in sys.path:
        sys.path.insert(0, str(search_path))

for p in [OUT_DIR, FIG_DIR, PAPER_ASSETS_DIR, PAPER_ASSETS_FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def save_figure(figure, path, **kwargs):
    """Save figures through a temporary file to avoid interrupted overwrites on Windows/Jupyter."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(f"{path.stem}.__tmp__{path.suffix}")
    figure.savefig(tmp_path, **kwargs)
    tmp_path.replace(path)

DATASET_PATH = DATA_DIR / "step1_simulation_dataset.csv"
TARGET = "eui_kwh_m2"   # Start with EUI.
RANDOM_SEED = 42
FAST_MODE = os.environ.get("EUI_FAST_MODE", "0") == "1"
N_JOBS = 1 if FAST_MODE else -1
BOOTSTRAP_N = 100 if FAST_MODE else 1000

<!-- CODEx bilingual cell explanation: start -->
### Cell 02 — 数据加载与特征工程 / Data loading and feature engineering

**中文说明**：读取 Step 1 仿真数据，构造朝向正弦/余弦、建筑占地面积、长宽比、窗型虚拟变量和 envelope_to_floor_ratio，并定义 39 个分析变量及其全称。该 cell 保证 SRC 输入与后续机器学习输入一致。

**输入与依赖**：读取上游步骤生成的 CSV、模型清单或配置对象，并检查必要字段是否存在。

**主要输出**：生成清洗后的 DataFrame、特征列表、训练测试划分或供后续单元复用的中间变量。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Loads the Step 1 simulation dataset, constructs orientation sine/cosine terms, footprint area, aspect ratio, window-type dummies, and envelope_to_floor_ratio, and defines the 39 analysis features with full labels. This keeps the SRC inputs aligned with the downstream machine-learning inputs.

**Inputs and dependencies**: Reads CSV files, model lists, or configuration objects produced by previous steps and validates required fields.

**Main outputs**: Creates cleaned DataFrames, feature lists, train/test splits, or intermediate objects reused downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 1) Load data ----------
assert DATASET_PATH.exists(), "Please run Step 1 first to generate step1_simulation_dataset.csv"
df = pd.read_csv(DATASET_PATH)
assert TARGET in df.columns, f"Target column missing from the dataset: {TARGET}"

# 1) Encode orientation as circular features.
df["orientation_sin"] = np.sin(np.deg2rad(df["orientation_deg"]))
df["orientation_cos"] = np.cos(np.deg2rad(df["orientation_deg"]))
df["footprint_area_m2"] = df["building_length"] * df["building_width"]
df["aspect_ratio"] = df["building_length"] / df["building_width"]

# 2) Convert window type into dummy variables.
df = pd.get_dummies(df, columns=["window_type_id"], prefix="window_type", drop_first=True)

# ---------- 1b) Geometry-based validation metrics ----------
df["building_height_calc_m"] = df["floor_num"] * df["floor_height"]

# Recover length and width from footprint area and aspect ratio.
L = np.sqrt(df["footprint_area_m2"] * df["aspect_ratio"])
W = np.sqrt(df["footprint_area_m2"] / df["aspect_ratio"])
P = 2 * (L + W)

# Envelope area divided by gross floor area (approximate).
df["envelope_to_floor_ratio"] = (
    P * df["building_height_calc_m"] + 2 * df["footprint_area_m2"]
) / df["gross_floor_area_m2"]

# 3) Final features used in this analysis step.
analysis_features = [
    'insul_thick', 'wwr', 'wall_thick',
    'u_win_n', 'u_win_s', 'u_win_e', 'u_win_w',
    'u_wall', 'u_roof', 'u_ground',
    'shgc_n', 'shgc_s', 'shgc_e', 'shgc_w',
    'roof_insul_thick',

    # Building geometry after reconstruction.
    'floor_num', 'footprint_area_m2', 'aspect_ratio', 'floor_height',
    'orientation_sin', 'orientation_cos',

    # Internal function and thermal-zone variables.
    'public_area', 'room_area', 'room_count',

    # Operation and system variables.
    'equip_power', 'dhw_per_person', 'occupancy_density', 'light_power',
    'cool_set', 'heat_set', 'dhw_temp',
    'cop_cooling', 'cop_heating', 'boiler_eff', 'fan_eff',
    'fresh_air_ach', 'operation_hours',

    # Window-type dummy variables.
    'window_type_2', 'window_type_3'
]

feature_groups = {
    "Envelope": [
        'insul_thick', 'wwr', 'wall_thick',
        'u_win_n', 'u_win_s', 'u_win_e', 'u_win_w',
        'u_wall', 'u_roof', 'u_ground',
        'shgc_n', 'shgc_s', 'shgc_e', 'shgc_w',
        'roof_insul_thick', 'window_type_2', 'window_type_3'
    ],
    "Geometry_Form": [
        'floor_num', 'footprint_area_m2', 'aspect_ratio', 'floor_height',
        'orientation_sin', 'orientation_cos'
    ],
    "Program_Zoning": [
        'public_area', 'room_area', 'room_count'
    ],
    "Operation_HVAC": [
        'equip_power', 'dhw_per_person', 'occupancy_density', 'light_power',
        'cool_set', 'heat_set', 'dhw_temp',
        'cop_cooling', 'cop_heating', 'boiler_eff', 'fan_eff',
        'fresh_air_ach', 'operation_hours'
    ]
}

feature_fullname_map = {
    "insul_thick": "External wall insulation layer thickness (m)",
    "wwr": "Overall window-to-wall ratio (-)",
    "wall_thick": "External wall structure thickness (m)",

    "u_win_n": "Heat transfer coefficient of north-facing windows (W/(m²·K))",
    "u_win_s": "Heat transfer coefficient of south-facing windows (W/(m²·K))",
    "u_win_e": "Heat transfer coefficient of east-facing windows (W/(m²·K))",
    "u_win_w": "Heat transfer coefficient of west-facing windows (W/(m²·K))",

    "u_wall": "Heat transfer coefficient of main external wall structure (W/(m²·K))",
    "u_roof": "Heat transfer coefficient of roof (W/(m²·K))",
    "u_ground": "Heat transfer coefficient of ground foundation (W/(m²·K))",

    "shgc_n": "Solar heat gain coefficient of north-facing windows (-)",
    "shgc_s": "Solar heat gain coefficient of south-facing windows (-)",
    "shgc_e": "Solar heat gain coefficient of east-facing windows (-)",
    "shgc_w": "Solar heat gain coefficient of west-facing windows (-)",

    "roof_insul_thick": "Roof insulation layer thickness (m)",

    "floor_num": "Total number of floors (floors)",
    "footprint_area_m2": "Building footprint area = main building length × main building width (m²)",
    "aspect_ratio": "Building length-to-width ratio (-)",
    "floor_height": "Standard floor height (m)",

    "orientation_sin": "Sine term of overall building orientation (-)",
    "orientation_cos": "Cosine term of overall building orientation (-)",

    "public_area": "Total area of public areas (m²)",
    "room_area": "Floor area of a single guest room (m²)",
    "room_count": "Total number of guest rooms (rooms)",

    "equip_power": "Indoor equipment power density (W/m²)",
    "dhw_per_person": "Daily domestic hot water consumption per capita (m³/(person·d))",
    "occupancy_density": "Personnel density (person/m²)",
    "light_power": "Indoor lighting power density (W/m²)",

    "cool_set": "Summer cooling set temperature (°C)",
    "heat_set": "Winter heating set temperature (°C)",
    "dhw_temp": "Domestic hot water supply temperature (°C)",

    "cop_cooling": "Rated COP of refrigeration unit (-)",
    "cop_heating": "COP of heat pump heating (-)",
    "boiler_eff": "Thermal efficiency of hot water boiler (-)",
    "fan_eff": "Efficiency of air conditioning fan (-)",
    "fresh_air_ach": "Fresh air exchange rate (times/h)",
    "operation_hours": "Total annual operating hours of system (h)",

    "window_type_2": "Window construction type dummy: Type 2 relative to Type 1 baseline (-)",
    "window_type_3": "Window construction type dummy: Type 3 relative to Type 1 baseline (-)",
}

X = df[analysis_features].copy()
y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Feature count:", len(analysis_features))

<!-- CODEx bilingual cell explanation: start -->
### Cell 03 — VIF 多重共线性检查 / VIF multicollinearity check

**中文说明**：逐变量用线性回归计算 VIF，并导出 vif_table.csv。该检查用于识别高度共线变量，避免 SRC 排名被冗余变量不合理放大或削弱。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Computes the variance inflation factor for each feature using auxiliary linear regressions and exports vif_table.csv. This identifies collinearity that could distort SRC ranking by redundant predictors.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 2) VIF check ----------
def compute_vif_table(X_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in X_df.columns:
        y_col = X_df[col].values
        X_other = X_df.drop(columns=[col]).values
        model = LinearRegression()
        model.fit(X_other, y_col)
        r2 = model.score(X_other, y_col)
        vif = np.inf if r2 >= 0.999999 else 1.0 / (1.0 - r2)
        rows.append({"feature": col, "VIF": vif})
    return pd.DataFrame(rows).sort_values("VIF", ascending=False)

vif_df = compute_vif_table(X)
vif_df["feature_full"] = vif_df["feature"].map(feature_fullname_map)
vif_df = vif_df[["feature", "feature_full", "VIF"]]

vif_df.to_csv(OUT_DIR / "vif_table.csv", index=False, encoding="utf-8-sig")
display(vif_df)

<!-- CODEx bilingual cell explanation: start -->
### Cell 04 — SRC bootstrap 函数 / SRC bootstrap function

**中文说明**：封装 SRC 估计过程：对 X 和 y 分别标准化、拟合线性回归、重复 bootstrap 重采样并计算 95% 置信区间和符号稳定性。该函数用于 EUI、OCEI 和物理验证的重复敏感性分析。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Wraps SRC estimation: standardises X and y, fits linear regression, performs bootstrap resampling, and computes 95% confidence intervals plus sign stability. The function is reused for EUI, OCEI, and physical-validation sensitivity analyses.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
def run_src_model(df_in, feature_list, target, seed=42, B=BOOTSTRAP_N):
    X = df_in[feature_list].copy()
    y = df_in[target].copy()

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_std = x_scaler.fit_transform(X)
    y_std = y_scaler.fit_transform(y.to_numpy().reshape(-1, 1)).ravel()

    model = LinearRegression()
    model.fit(X_std, y_std)
    coef_full = model.coef_

    # bootstrap
    rng = np.random.default_rng(seed)
    coef_boot = np.zeros((B, X.shape[1]))

    for b in range(B):
        idx = rng.integers(0, len(X), len(X))
        Xb = X.iloc[idx].reset_index(drop=True)
        yb = y.iloc[idx].reset_index(drop=True)

        Xb_std = StandardScaler().fit_transform(Xb)
        yb_std = StandardScaler().fit_transform(yb.to_numpy().reshape(-1, 1)).ravel()

        mb = LinearRegression()
        mb.fit(Xb_std, yb_std)
        coef_boot[b, :] = mb.coef_

    out = pd.DataFrame({
        "feature": feature_list,
        "SRC": coef_full,
        "abs_SRC": np.abs(coef_full),
        "CI_low": np.quantile(coef_boot, 0.025, axis=0),
        "CI_high": np.quantile(coef_boot, 0.975, axis=0),
    })

    out["sign_stable"] = (
        (out["CI_low"] > 0) | (out["CI_high"] < 0)
    )

    return out.sort_values("abs_SRC", ascending=False).reset_index(drop=True)

<!-- CODEx bilingual cell explanation: start -->
### Cell 05 — 线性近似交叉验证 / Cross-validation of the linear approximation

**中文说明**：用 5 折交叉验证评估线性回归对 EUI 的拟合能力，输出 CV R2 和 RMSE。该结果界定 SRC 作为线性解释方法的适用边界。

**输入与依赖**：依赖上游特征矩阵、目标变量、候选模型或交叉验证设置。

**主要输出**：输出拟合模型、评价指标、交叉验证结果、预测值或模型参数表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Evaluates the linear regression approximation for EUI using 5-fold cross-validation and reports CV R2 and RMSE. The result defines the validity boundary for using SRC as a linear interpretability method.

**Inputs and dependencies**: Depends on upstream feature matrices, target values, candidate models, or cross-validation settings.

**Main outputs**: Produces fitted models, evaluation metrics, cross-validation results, predictions, or parameter tables.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 3) Check usability of the linear approximation ----------
def cv_rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
X_std = StandardScaler().fit_transform(X)

pred_cv = cross_val_predict(
    LinearRegression(),
    X_std,
    y,
    cv=cv
)

print({
    "cv_r2": round(r2_score(y, pred_cv), 4),
    "cv_rmse": round(cv_rmse(y, pred_cv), 4),
})

<!-- CODEx bilingual cell explanation: start -->
### Cell 06 — 全变量 SRC 与置信区间 / Full-feature SRC with confidence intervals

**中文说明**：对 39 个输入变量计算 bootstrap SRC、绝对 SRC、95% 置信区间和符号稳定性，并保存 src_indices_bootstrap.csv。该表是变量筛选和后续 18 变量建模的主证据。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Computes bootstrap SRC, absolute SRC, 95% confidence intervals, and sign stability for all 39 input variables, then saves src_indices_bootstrap.csv. This table is the primary evidence for variable selection and downstream 18-feature modelling.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 4) SRC with bootstrap resampling ----------
def fit_src(X_df: pd.DataFrame, y_series: pd.Series):
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_std = x_scaler.fit_transform(X_df)
    y_std = y_scaler.fit_transform(y_series.to_numpy().reshape(-1, 1)).ravel()

    model = LinearRegression()
    model.fit(X_std, y_std)
    return model.coef_

# SRC fitted on all samples.
coef_full = fit_src(X, y)

# Bootstrap
B = BOOTSTRAP_N
rng = np.random.default_rng(RANDOM_SEED)
coef_boot = np.zeros((B, X.shape[1]))

for b in range(B):
    idx = rng.integers(0, len(X), len(X))
    Xb = X.iloc[idx].reset_index(drop=True)
    yb = y.iloc[idx].reset_index(drop=True)
    coef_boot[b, :] = fit_src(Xb, yb)

src_df = pd.DataFrame({
    "feature": X.columns,
    "SRC": coef_full,
    "abs_SRC": np.abs(coef_full),
    "CI_low": np.quantile(coef_boot, 0.025, axis=0),
    "CI_high": np.quantile(coef_boot, 0.975, axis=0),
})

src_df["sign_stable"] = (
    (src_df["CI_low"] > 0) | (src_df["CI_high"] < 0)
)

src_df = src_df.sort_values("abs_SRC", ascending=False).reset_index(drop=True)
src_df["feature_full"] = src_df["feature"].map(feature_fullname_map)

src_df = src_df[
    ["feature", "feature_full", "SRC", "abs_SRC", "CI_low", "CI_high", "sign_stable"]
]

src_df.to_csv(OUT_DIR / "src_indices_bootstrap.csv", index=False, encoding="utf-8-sig")
display(src_df.head(20))

<!-- CODEx bilingual cell explanation: start -->
### Cell 07 — SHAP 非线性敏感性补充 / SHAP nonlinear sensitivity supplement

**中文说明**：训练 XGBoost 并计算 SHAP 平均绝对贡献，将非线性重要性排序与 SRC 排序比较，输出 Spearman 秩相关、Top-18 Jaccard 重叠、并排图和 beeswarm 图。该 cell 用非线性解释方法验证 SRC 排名的稳健性。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Trains XGBoost, computes mean absolute SHAP values, and compares nonlinear importance ranking with SRC ranking. It reports Spearman rank correlation, Top-18 Jaccard overlap, side-by-side importance plots, and a SHAP beeswarm plot. This cell validates the robustness of SRC ranking using a nonlinear interpretability method.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ============================================================
# SHAP-based sensitivity analysis
#
# Supplement SRC (linear) with SHAP values from XGBoost to:
# 1. Address the SRC-linearity limitation (Reviewer 1 Major 3)
# 2. Validate the 18-variable cutoff using a nonlinear method
# 3. Detect interaction effects invisible to SRC
# ============================================================

import shap
from xgboost import XGBRegressor

# Train XGBoost on all 39 features to get SHAP values
xgb = XGBRegressor(
    n_estimators=(200 if FAST_MODE else 500), max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=N_JOBS
)
xgb.fit(X, y)

# Compute SHAP values
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X)

# SHAP feature importance (mean |SHAP|)
shap_importance = pd.DataFrame({
    'feature': X.columns,
    'shap_abs_mean': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_abs_mean', ascending=False).reset_index(drop=True)

# Merge with SRC results for comparison
compare_df = src_df[['feature', 'abs_SRC', 'SRC']].merge(
    shap_importance, on='feature', how='left'
)
compare_df['src_rank'] = compare_df['abs_SRC'].rank(ascending=False)
compare_df['shap_rank'] = compare_df['shap_abs_mean'].rank(ascending=False)
compare_df['rank_delta'] = compare_df['src_rank'] - compare_df['shap_rank']

print("=" * 50)
print("SRC vs SHAP VARIABLE RANKING COMPARISON")
print("=" * 50)
print(f"Spearman rank correlation: {compare_df['src_rank'].corr(compare_df['shap_rank'], method='spearman'):.4f}")
print(f"\nTop 10 variables by SRC:  {compare_df.nsmallest(10, 'src_rank')['feature'].tolist()}")
print(f"Top 10 variables by SHAP: {compare_df.nsmallest(10, 'shap_rank')['feature'].tolist()}")
print(f"\nVariables with largest rank shift (|delta| > 5):")
large_shift = compare_df[compare_df['rank_delta'].abs() > 5].sort_values('rank_delta', key=abs, ascending=False)
for _, row in large_shift.iterrows():
    direction = 'higher in SHAP' if row['rank_delta'] < 0 else 'higher in SRC'
    print(f"  {row['feature']}: SRC rank={int(row['src_rank'])}, SHAP rank={int(row['shap_rank'])} ({direction})")

# Check: Do all SRC top-18 appear in SHAP top-18?
src_top18 = set(compare_df.nsmallest(18, 'src_rank')['feature'])
shap_top18 = set(compare_df.nsmallest(18, 'shap_rank')['feature'])
overlap = src_top18 & shap_top18
only_src = src_top18 - shap_top18
only_shap = shap_top18 - src_top18
print(f"\nTop-18 overlap: {len(overlap)}/18 (Jaccard={len(overlap)/len(src_top18|shap_top18):.3f})")
if only_src:
    print(f"Only in SRC top-18: {only_src}")
if only_shap:
    print(f"Only in SHAP top-18: {only_shap}")

# Visualisation: Side-by-side SRC vs SHAP
fig, axes = plt.subplots(1, 2, figsize=(16, 10), dpi=150)

# Panel 1: SRC top 20
top20_src = compare_df.nsmallest(20, 'src_rank').sort_values('abs_SRC')
axes[0].barh(top20_src['feature'], top20_src['abs_SRC'], color='steelblue')
axes[0].set_xlabel('|SRC| (Standardised Regression Coefficient)', fontsize=12)
axes[0].set_title('Top 20 variables ranked by SRC (linear)', fontsize=13)
axes[0].grid(axis='x', alpha=0.3)

# Panel 2: SHAP top 20
top20_shap = compare_df.nsmallest(20, 'shap_rank').sort_values('shap_abs_mean')
bar_colors = ['darkorange' if f in only_shap else 'steelblue' for f in top20_shap['feature']]
axes[1].barh(top20_shap['feature'], top20_shap['shap_abs_mean'], color=bar_colors)
axes[1].set_xlabel('Mean |SHAP Value| (XGBoost)', fontsize=12)
axes[1].set_title('Top 20 variables ranked by SHAP (nonlinear)', fontsize=13)
axes[1].grid(axis='x', alpha=0.3)
# Highlight variables not in SRC top-18
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='In both SRC & SHAP top-18'),
                   Patch(facecolor='darkorange', label='SHAP-only (not in SRC top-18)')]
axes[1].legend(handles=legend_elements, fontsize=9, loc='lower right')

plt.tight_layout()
out_dir = PROJECT_ROOT / 'outputs_step2' / 'figures'
out_dir.mkdir(parents=True, exist_ok=True)
save_figure(fig, out_dir / 'src_vs_shap_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# SHAP summary plot (beeswarm)
fig2, ax2 = plt.subplots(figsize=(10, 8), dpi=150)
shap.summary_plot(shap_values, X, show=False, max_display=20)
plt.tight_layout()
save_figure(fig2, out_dir / 'shap_summary_beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()

# Save comparison table
compare_df.to_csv(PROJECT_ROOT / 'outputs_step2' / 'src_shap_ranking_comparison.csv', index=False)
print("\nSRC-SHAP comparison saved.")


<!-- CODEx 2026-06-16 reviewer-strengthening: SHAP-SRC 分歧变量深度分析 -->
### SHAP-SRC 分歧变量深度分析

该分析从“排序相关”推进到“分歧是否改变预测结论”：若 SHAP 独有变量加入 SRC Top-18 后增量 R2 很小，则说明 SRC 未遗漏关键预测变量。

In [ ]:

# ============================================================
# 2026-06-16 V1: SHAP-SRC divergence and incremental-value analysis
# Tests whether SHAP-only variables materially improve prediction.
# ============================================================

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

src_top18_list = compare_df.nsmallest(18, "src_rank")["feature"].tolist()
shap_top18_list = compare_df.nsmallest(18, "shap_rank")["feature"].tolist()
src_top18 = set(src_top18_list)
shap_top18 = set(shap_top18_list)
only_src = sorted(src_top18 - shap_top18)
only_shap = sorted(shap_top18 - src_top18)

divergence_rows = []
base_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RidgeCV(alphas=np.logspace(-3, 4, 40), cv=5)),
])
cv = KFold(n_splits=min(5, len(df)), shuffle=True, random_state=42)
base_score = cross_val_score(base_pipe, X[src_top18_list], y, cv=cv, scoring="r2", n_jobs=N_JOBS).mean()

for feature in only_shap:
    add_features = src_top18_list + [feature]
    score = cross_val_score(base_pipe, X[add_features], y, cv=cv, scoring="r2", n_jobs=N_JOBS).mean()
    src_rank_val = compare_df.loc[compare_df["feature"] == feature, "src_rank"].iloc[0]
    shap_rank_val = compare_df.loc[compare_df["feature"] == feature, "shap_rank"].iloc[0]
    divergence_rows.append({
        "feature": feature,
        "src_rank": src_rank_val,
        "shap_rank": shap_rank_val,
        "ridgecv_r2_base_src18": base_score,
        "ridgecv_r2_with_feature": score,
        "incremental_r2": score - base_score,
        "interpretation": "SHAP-only variable; tests whether nonlinear importance implies omitted predictive value",
    })

for feature in only_src:
    src_rank_val = compare_df.loc[compare_df["feature"] == feature, "src_rank"].iloc[0]
    shap_rank_val = compare_df.loc[compare_df["feature"] == feature, "shap_rank"].iloc[0]
    divergence_rows.append({
        "feature": feature,
        "src_rank": src_rank_val,
        "shap_rank": shap_rank_val,
        "ridgecv_r2_base_src18": base_score,
        "ridgecv_r2_with_feature": np.nan,
        "incremental_r2": np.nan,
        "interpretation": "SRC-only variable; mostly reflects stable linear main-effect contribution",
    })

divergence_df = pd.DataFrame(divergence_rows).sort_values(["interpretation", "incremental_r2"], ascending=[True, False])
divergence_df.to_csv(OUT_DIR / "shap_src_divergence_incremental_r2.csv", index=False, encoding="utf-8-sig")

plot_features = only_shap[:4] if only_shap else shap_top18_list[:4]
fig, axes = plt.subplots(1, max(1, len(plot_features)), figsize=(4.2 * max(1, len(plot_features)), 3.8), dpi=150, constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, feature in zip(axes, plot_features):
    idx = list(X.columns).index(feature)
    ax.scatter(X[feature], shap_values[:, idx], c=df["eui_kwh_m2"], cmap="viridis", s=22, alpha=0.78, edgecolor="none")
    ax.axhline(0, color="#6B7280", linewidth=0.8, linestyle="--")
    ax.set_xlabel(feature)
    ax.set_ylabel("SHAP value")
    ax.set_title(f"{feature}\nSRC rank {int(compare_df.loc[compare_df['feature']==feature,'src_rank'].iloc[0])}, "
                 f"SHAP rank {int(compare_df.loc[compare_df['feature']==feature,'shap_rank'].iloc[0])}")
save_figure(fig, FIG_DIR / "shap_divergence_dependence.png", dpi=300, bbox_inches="tight")
plt.show()

print("SHAP-SRC divergence and incremental-R2 analysis saved:", OUT_DIR / "shap_src_divergence_incremental_r2.csv")
print(f"SRC Top-18 baseline RidgeCV R2: {base_score:.4f}")
display(divergence_df.round(5))


<!-- CODEx bilingual cell explanation: start -->
### Cell 08 — 18 变量截断阈值分析 / Eighteen-variable cutoff analysis

**中文说明**：绘制 SRC 碎石图、累计贡献曲线和交叉验证 R2 随变量数变化曲线，并导出 cv_r2_by_variable_count.csv。该 cell 用三条独立证据线说明为什么选 18 个变量而不是任意截断。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Plots the SRC scree curve, cumulative contribution curve, and cross-validated R2 as a function of feature count, then exports cv_r2_by_variable_count.csv. It provides three independent lines of evidence for choosing 18 variables rather than an arbitrary cutoff.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ============================================================
# Variable-selection cutoff analysis
#
# Analyse the natural breakpoints in SRC distribution to justify
# the 18-variable cutoff rather than an arbitrary threshold.
# ============================================================

# Compute cumulative SRC contribution and incremental drops
src_sorted = src_df.sort_values('abs_SRC', ascending=False).reset_index(drop=True)
src_sorted['cumulative_abs_src'] = src_sorted['abs_SRC'].cumsum()
src_sorted['cumulative_pct'] = (src_sorted['cumulative_abs_src'] /
                                 src_sorted['abs_SRC'].sum() * 100)
src_sorted['incremental_src'] = src_sorted['abs_SRC'].diff().abs()
src_sorted['pct_of_max'] = src_sorted['abs_SRC'] / src_sorted['abs_SRC'].max() * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150)

# Panel 1: SRC scree plot
ax = axes[0]
colors = ['darkred' if i < 18 else 'steelblue' for i in range(len(src_sorted))]
ax.bar(range(1, len(src_sorted)+1), src_sorted['abs_SRC'], color=colors, width=0.7)
ax.axvline(18.5, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(19, ax.get_ylim()[1]*0.95, f'Cutoff at 18 vars', fontsize=9, ha='left')
ax.set_xlabel('Variable rank by |SRC|', fontsize=12)
ax.set_ylabel('|SRC|', fontsize=12)
ax.set_title('Scree plot of absolute SRC values', fontsize=13)
ax.grid(axis='y', alpha=0.3)

# Panel 2: Cumulative contribution
ax = axes[1]
ax.plot(range(1, len(src_sorted)+1), src_sorted['cumulative_pct'], 'o-',
        color='darkred', markersize=4, linewidth=1.5)
ax.axhline(90, color='grey', linestyle=':', alpha=0.5, label='90% cumulative |SRC|')
ax.axhline(95, color='grey', linestyle='--', alpha=0.5, label='95% cumulative |SRC|')
ax.axvline(18, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax.fill_between([1, 18], 0, 100, alpha=0.05, color='green')
ax.text(10, 50, f'First 18 vars:\\n{src_sorted.iloc[17]["cumulative_pct"]:.1f}% of total |SRC|',
        fontsize=9, ha='center')
ax.set_xlabel('Number of included variables', fontsize=12)
ax.set_ylabel('Cumulative |SRC| (%)', fontsize=12)
ax.set_title('Cumulative SRC contribution', fontsize=13)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 3: CV R2 vs number of top variables
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

cv_r2_by_n = []
for n in range(1, len(X.columns)+1):
    top_n_features = src_sorted.head(n)['feature'].tolist()
    X_n = X[top_n_features]
    cv_scores = cross_val_score(LinearRegression(), X_n, y, cv=5, scoring='r2')
    cv_r2_by_n.append({'n_vars': n, 'cv_r2_mean': cv_scores.mean(), 'cv_r2_std': cv_scores.std()})

cv_df = pd.DataFrame(cv_r2_by_n)

ax = axes[2]
ax.fill_between(cv_df['n_vars'],
                cv_df['cv_r2_mean'] - cv_df['cv_r2_std'],
                cv_df['cv_r2_mean'] + cv_df['cv_r2_std'],
                alpha=0.2, color='steelblue')
ax.plot(cv_df['n_vars'], cv_df['cv_r2_mean'], 'o-', color='steelblue', markersize=4, linewidth=1.5)
ax.axvline(18, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Number of included variables', fontsize=12)
ax.set_ylabel('5-Fold CV R² (Linear Regression)', fontsize=12)
ax.set_title('Predictive performance versus number of variables', fontsize=13)
ax.grid(alpha=0.3)

# Annotate key points
best_n = cv_df.loc[cv_df['cv_r2_mean'].idxmax()]
ax.annotate(f'Peak at n={int(best_n["n_vars"])}\\nR²={best_n["cv_r2_mean"]:.4f}',
            xy=(best_n['n_vars'], best_n['cv_r2_mean']),
            xytext=(best_n['n_vars']+5, best_n['cv_r2_mean']-0.02),
            arrowprops=dict(arrowstyle='->', color='darkred'),
            fontsize=9, color='darkred')

plt.tight_layout()
save_figure(fig, PROJECT_ROOT / 'outputs_step2' / 'figures' / 'variable_selection_analysis.png',
            dpi=300, bbox_inches='tight')
plt.show()

print("\nVARIABLE SELECTION ANALYSIS")
print(f"Cumulative |SRC| at n=10: {src_sorted.iloc[9]['cumulative_pct']:.1f}%")
print(f"Cumulative |SRC| at n=18: {src_sorted.iloc[17]['cumulative_pct']:.1f}%")
print(f"Cumulative |SRC| at n=25: {src_sorted.iloc[24]['cumulative_pct']:.1f}%")
print(f"CV R² with 18 vars: {cv_df.loc[cv_df['n_vars']==18, 'cv_r2_mean'].values[0]:.4f}")
print(f"Max CV R² ({cv_df['cv_r2_mean'].max():.4f}) at n={int(best_n['n_vars'])}")

# Save
cv_df.to_csv(PROJECT_ROOT / 'outputs_step2' / 'cv_r2_by_variable_count.csv', index=False)


<!-- CODEx 2026-06-16 reviewer-strengthening: 第 15-18 名变量边际消融检验 -->
### 第 15-18 名变量边际消融检验

该分析检验 18 变量截断是否保守。若逐个移除边际变量后交叉验证 R2 变化很小，则说明第 18 名并非脆弱阈值，而是兼顾解释充分性与实用性的保守选择。

In [ ]:

# ============================================================
# 2026-06-16 V1: marginal-variable ablation for variables ranked 15-18
# Tests whether the 18-variable cutoff is conservative rather than fragile.
# ============================================================

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

top18_ordered = src_sorted.head(18)["feature"].tolist()
marginal_features = top18_ordered[14:18]
cv = KFold(n_splits=min(5, len(df)), shuffle=True, random_state=42)
lin_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
base_r2 = cross_val_score(lin_pipe, X[top18_ordered], y, cv=cv, scoring="r2", n_jobs=N_JOBS).mean()

rows = [{
    "case": "all_top18",
    "removed_feature": "",
    "n_features": len(top18_ordered),
    "cv_r2_mean": base_r2,
    "delta_r2_vs_top18": 0.0,
    "physical_role": "Full 18-variable baseline",
}]

role_map = {
    "equip_power": "Equipment power density; proxy for plug-load internal gains and electricity demand",
    "heat_set": "Heating setpoint; proxy for winter operational-control boundary",
    "room_area": "Guest-room area; proxy for functional-space scale",
    "u_wall": "External-wall U-value; proxy for opaque-envelope heat transfer",
}

for feature in marginal_features:
    remaining = [f for f in top18_ordered if f != feature]
    score = cross_val_score(lin_pipe, X[remaining], y, cv=cv, scoring="r2", n_jobs=N_JOBS).mean()
    rows.append({
        "case": f"remove_{feature}",
        "removed_feature": feature,
        "n_features": len(remaining),
        "cv_r2_mean": score,
        "delta_r2_vs_top18": score - base_r2,
        "physical_role": role_map.get(feature, "Marginal variable"),
    })

remaining_14 = [f for f in top18_ordered if f not in marginal_features]
score_14 = cross_val_score(lin_pipe, X[remaining_14], y, cv=cv, scoring="r2", n_jobs=N_JOBS).mean()
rows.append({
    "case": "remove_rank15_to_18",
    "removed_feature": ", ".join(marginal_features),
    "n_features": len(remaining_14),
    "cv_r2_mean": score_14,
    "delta_r2_vs_top18": score_14 - base_r2,
    "physical_role": "Conservative test removing all rank-15 to rank-18 marginal variables",
})

ablation_df = pd.DataFrame(rows)
ablation_df.to_csv(OUT_DIR / "marginal_variable_ablation.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8.8, 4.8), dpi=150)
plot_df = ablation_df[ablation_df["case"] != "all_top18"].copy()
colors = ["#4C78A8" if v >= -0.01 else "#B64646" for v in plot_df["delta_r2_vs_top18"]]
ax.barh(plot_df["case"], plot_df["delta_r2_vs_top18"], color=colors)
ax.axvline(0, color="#6B7280", linewidth=0.9)
ax.axvline(-0.01, color="#F58518", linestyle="--", linewidth=1.0, label="ΔR2 = -0.01")
ax.set_xlabel("Change in 5-fold CV R2 relative to the Top-18 baseline")
ax.set_title("One-at-a-time ablation of rank-15 to rank-18 marginal variables")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
save_figure(fig, FIG_DIR / "marginal_variable_ablation.png", dpi=300, bbox_inches="tight")
plt.show()

print("Marginal-variable ablation analysis saved:", OUT_DIR / "marginal_variable_ablation.csv")
display(ablation_df.round(5))


### SRC 方法的线性假设局限与补充验证

**针对审稿人关于 SRC 线性假设的回应：**

标准化回归系数（SRC）本质上基于普通最小二乘（OLS）回归，其有效性与模型的线性拟合优度密切相关。当响应面存在显著非线性时，SRC 可能低估通过交互作用或阈值效应产生影响的关键变量。

为评估 SRC 在当前数据集上的适用性并补充非线性视角，本研究增加了以下分析：

1. **SHAP 值排序验证**：使用 XGBoost（一种基于树的非线性模型）计算 SHAP（SHapley Additive exPlanations）值，与 SRC 排序进行交叉验证。Spearman 秩相关系数衡量两种方法排序的一致性。

2. **累积贡献分析**：绘制 SRC 绝对值从大到小的累积贡献曲线，识别自然断点，为 18 变量截断阈值提供定量依据。

3. **预测能力曲线**：通过 5 折交叉验证的线性回归 R² 随变量数量变化曲线，评估增加变量对预测能力的边际贡献，确定收益递减的拐点。


<!-- CODEx bilingual cell explanation: start -->
### Cell 09 — SRC 分组汇总 / Group-level SRC summary

**中文说明**：按围护结构、几何形态、功能分区和运行/HVAC 分组汇总绝对 SRC、平均绝对 SRC 和符号稳定变量数量。该结果帮助把单变量排序转化为论文讨论中的系统层面解释。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Aggregates absolute SRC, mean absolute SRC, and sign-stable counts by envelope, geometry/form, program/zoning, and operation/HVAC groups. This translates feature-level ranking into system-level interpretation for the manuscript discussion.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 5) Group-level summary ----------
group_rows = []
for gname, feats in feature_groups.items():
    sub = src_df[src_df["feature"].isin(feats)].copy()
    group_rows.append({
        "group": gname,
        "n_features": len(feats),
        "sum_abs_SRC": sub["abs_SRC"].sum(),
        "mean_abs_SRC": sub["abs_SRC"].mean(),
        "n_sign_stable": int(sub["sign_stable"].sum())
    })

group_df = pd.DataFrame(group_rows).sort_values("sum_abs_SRC", ascending=False)
group_df.to_csv(OUT_DIR / "src_group_summary.csv", index=False, encoding="utf-8-sig")
display(group_df)

<!-- CODEx bilingual cell explanation: start -->
### Cell 10 — 全输入变量 Top-18 标记图 / All-input SRC plot with Top-18 markers

**中文说明**：重新计算当前输入变量 SRC 并用颜色标记 Top-18 变量，保存全变量横向条形图。该图用于直观看到第 18 名之后 SRC 是否趋于平缓。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。

**主要输出**：导出论文或复核用图像文件，并在必要时同步导出支撑图表的数据表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Recomputes SRC for the current input set and marks the Top-18 variables in a horizontal bar plot. The figure visually checks whether SRC magnitudes flatten after the 18th variable.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow with a consistent plotting style.

**Main outputs**: Exports manuscript or audit figures and, when needed, the source tables behind the figure.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
src_current = run_src_model(df, analysis_features, TARGET, seed=42, B=BOOTSTRAP_N)
src_current["rank_abs"] = src_current["abs_SRC"].rank(method="first", ascending=False)
src_current["is_top18"] = src_current["rank_abs"] <= 18

src_current["feature_full"] = src_current["feature"].map(feature_fullname_map)

src_current.to_csv(OUT_DIR / "src_current_all_inputs.csv", index=False, encoding="utf-8-sig")

plot_df = src_current.sort_values("abs_SRC", ascending=True).copy()
colors = ["tab:red" if x else "tab:blue" for x in plot_df["is_top18"]]

fig, ax = plt.subplots(figsize=(12.5, 12.5))
ax.barh(plot_df["feature"], plot_df["abs_SRC"], color=colors)
ax.set_title("SRC of ALL input variables")
ax.set_xlabel("|SRC|")
ax.set_ylabel("Feature")
fig.tight_layout()
save_figure(fig, FIG_DIR / "src_all_current_inputs_top18_marked.png", dpi=300, bbox_inches="tight")
plt.show()

<!-- CODEx bilingual cell explanation: start -->
### Cell 11 — Top-18 SRC 方向图 / Top-18 SRC direction plot

**中文说明**：绘制前 18 个变量的有符号 SRC，展示变量对 EUI 的正向或负向影响。该图支撑论文中关于 DHW、楼层数、房间数等关键变量作用方向的解释。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。

**主要输出**：导出论文或复核用图像文件，并在必要时同步导出支撑图表的数据表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Plots signed SRC values for the top 18 variables, showing whether each feature increases or decreases EUI. The figure supports manuscript interpretation of key variables such as DHW, floor number, and room count.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow with a consistent plotting style.

**Main outputs**: Exports manuscript or audit figures and, when needed, the source tables behind the figure.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 6) Top SRC ----------
plot_df = src_df.head(18).iloc[::-1].copy()

fig, ax = plt.subplots(figsize=(8.8, 7.0))
ax.barh(plot_df["feature"], plot_df["SRC"])
ax.axvline(0, color="black", linewidth=1.0)
ax.set_title("Top 18 standardized regression coefficients (SRC)")
ax.set_xlabel("SRC")
fig.tight_layout()
save_figure(fig, FIG_DIR / "src_top18.png", bbox_inches="tight")
plt.show()

<!-- CODEx bilingual cell explanation: start -->
### Cell 12 — SRC 符号稳定性图 / SRC sign-stability plot

**中文说明**：用颜色区分 bootstrap 置信区间是否跨越 0，展示 Top-18 变量的符号稳定性。该结果防止把统计不稳定的弱变量解释为可靠结论。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。

**主要输出**：导出论文或复核用图像文件，并在必要时同步导出支撑图表的数据表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Uses colour to indicate whether bootstrap confidence intervals cross zero for the Top-18 variables. This prevents statistically unstable weak predictors from being overinterpreted as robust findings.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow with a consistent plotting style.

**Main outputs**: Exports manuscript or audit figures and, when needed, the source tables behind the figure.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 7) Stability markers ----------
sel = src_df.head(18).copy().sort_values("abs_SRC", ascending=True)

fig, ax = plt.subplots(figsize=(8.8, 7.0))
colors = ["tab:red" if s else "blue" for s in sel["sign_stable"]]
ax.barh(sel["feature"], sel["abs_SRC"], color=colors)
ax.set_title("Top 18 |SRC| (red = sign-stable; blue = sign-unstable)")
ax.set_xlabel("|SRC|")
fig.tight_layout()
save_figure(fig, FIG_DIR / "src_abs_top18_stability.png", bbox_inches="tight")
plt.show()

<!-- CODEx bilingual cell explanation: start -->
### Cell 13 — 几何物理关系散点检查 / Geometry-physics scatter checks

**中文说明**：绘制 envelope_to_floor_ratio、floor_num 与 EUI/site energy 的关系，用物理直觉检查几何变量对面积归一化指标和总能耗指标的不同影响。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Plots relationships between envelope_to_floor_ratio, floor_num, and EUI/site energy to check whether geometry variables behave consistently with physical intuition for area-normalised and total-energy metrics.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].scatter(df["envelope_to_floor_ratio"], df["eui_kwh_m2"], alpha=0.7)
axes[0].set_xlabel("envelope_to_floor_ratio")
axes[0].set_ylabel("eui_kwh_m2")
axes[0].set_title("Envelope area ratio versus EUI")

axes[1].scatter(df["floor_num"], df["eui_kwh_m2"], alpha=0.7)
axes[1].set_xlabel("floor_num")
axes[1].set_ylabel("eui_kwh_m2")
axes[1].set_title("Number of floors versus EUI")

axes[2].scatter(df["floor_num"], df["site_energy_kwh"], alpha=0.7)
axes[2].set_xlabel("floor_num")
axes[2].set_ylabel("site_energy_kwh")
axes[2].set_title("Number of floors versus total energy")

plt.tight_layout()
plt.show()

<!-- CODEx bilingual cell explanation: start -->
### Cell 14 — EUI 与总能耗 SRC 对比 / SRC comparison for EUI and site energy

**中文说明**：分别以 EUI 和 site_energy_kwh 为目标重新计算 SRC，并重点展示几何与功能变量的差异。该对比说明面积归一化会改变变量作用方向和大小。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Recomputes SRC using EUI and site_energy_kwh as separate targets and highlights differences for geometry and program variables. The comparison shows how area normalisation changes effect direction and magnitude.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
src_eui = run_src_model(df, analysis_features, "eui_kwh_m2", seed=42, B=BOOTSTRAP_N)
src_site = run_src_model(df, analysis_features, "site_energy_kwh", seed=42, B=BOOTSTRAP_N)

compare_df = pd.DataFrame({
    "feature": analysis_features,
    "SRC_EUI": src_eui.set_index("feature").reindex(analysis_features)["SRC"].values,
    "SRC_site_energy": src_site.set_index("feature").reindex(analysis_features)["SRC"].values,
}).sort_values("SRC_EUI", key=np.abs, ascending=False)

display(compare_df.loc[
    compare_df["feature"].isin([
        "floor_num", "footprint_area_m2", "floor_height",
        "aspect_ratio", "public_area", "room_area", "room_count"
    ])
])
compare_df.to_csv(OUT_DIR / "src_target_comparison.csv", index=False, encoding="utf-8-sig")

<!-- CODEx bilingual cell explanation: start -->
### Cell 15 — 物理扩展变量验证 / Physics-extended variable validation

**中文说明**：把 envelope_to_floor_ratio 纳入物理扩展变量集重新运行 SRC，验证主要结论是否依赖原始几何变量表达方式。结果导出为 src_physical_validation.csv。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Adds envelope_to_floor_ratio to a physics-extended feature set and reruns SRC to test whether the main conclusions depend on the original geometry representation. Results are exported as src_physical_validation.csv.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
physics_features = [
    'insul_thick', 'wwr', 'wall_thick',
    'u_win_n', 'u_win_s', 'u_win_e', 'u_win_w',
    'u_wall', 'u_roof', 'u_ground',
    'shgc_n', 'shgc_s', 'shgc_e', 'shgc_w',
    'roof_insul_thick',
    'envelope_to_floor_ratio',
    'public_area', 'room_area', 'room_count',
    'equip_power', 'dhw_per_person', 'occupancy_density', 'light_power',
    'cool_set', 'heat_set', 'dhw_temp',
    'cop_cooling', 'cop_heating', 'boiler_eff', 'fan_eff',
    'fresh_air_ach', 'operation_hours',
    'orientation_sin', 'orientation_cos',
    'window_type_2', 'window_type_3'
]

src_eui_physics = run_src_model(df, physics_features, "eui_kwh_m2", seed=42, B=BOOTSTRAP_N)
display(src_eui_physics.head(15))

src_eui_physics.to_csv(
    OUT_DIR / "src_physical_validation.csv",
    index=False,
    encoding="utf-8-sig"
)